# 📧 AI Email Assistant & Reply Generator

A beginner-friendly Generative AI project that drafts new emails and suggests replies with a selected tone and length.

**What this notebook demonstrates**
- Email drafting and reply generation
- Tone and length customization
- Prompt construction for an LLM
- A demo mode that works without an API key
- Optional live generation using an OpenAI-compatible API
- Basic validation and human review before sending

> **Privacy note:** Use sample or anonymized emails. This notebook generates drafts only; it does not send emails.

## 1. Install dependencies

Run this cell once in your notebook environment. If the packages are already installed, you can skip it.

In [ ]:
%pip install -q openai python-dotenv pandas

## 2. Imports and configuration

In [ ]:
import os
from getpass import getpass
from typing import Optional

import pandas as pd

# Optional: set OPENAI_API_KEY in your environment before running this notebook.
# You may also enter the key interactively later. Do not hard-code or commit secrets.
API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

print("API key configured:", bool(API_KEY))
print("Model:", MODEL_NAME)

## 3. Define the email assistant

The prompt asks the model to preserve the facts supplied by the user, avoid inventing commitments, and return only a draft.

In [ ]:
ALLOWED_TONES = {
    "Professional", "Friendly", "Formal", "Concise", "Appreciative", "Persuasive"
}
ALLOWED_LENGTHS = {"Short", "Standard", "Detailed"}

def build_email_prompt(
    mode: str,
    context: str,
    tone: str = "Professional",
    length: str = "Standard",
    subject: Optional[str] = None,
    key_points: Optional[str] = None,
) -> str:
    """Build a structured prompt for email drafting or reply generation."""
    if mode not in {"Draft an Email", "Generate a Reply"}:
        raise ValueError("mode must be 'Draft an Email' or 'Generate a Reply'")
    if not context or not context.strip():
        raise ValueError("Please provide email context or the incoming email text.")
    if tone not in ALLOWED_TONES:
        raise ValueError(f"tone must be one of: {', '.join(sorted(ALLOWED_TONES))}")
    if length not in ALLOWED_LENGTHS:
        raise ValueError(f"length must be one of: {', '.join(sorted(ALLOWED_LENGTHS))}")

    points = key_points.strip() if key_points else "No additional points supplied."
    subject_text = subject.strip() if subject else "Suggest a subject if this is a new email."

    return f"""You are a careful email-writing assistant.
Task: {mode}
Tone: {tone}
Desired length: {length}
Subject instruction: {subject_text}

User-provided context or incoming email:
<context>
{context.strip()}
</context>

Additional points the user wants to communicate:
<key_points>
{points}
</key_points>

Rules:
- Draft an email, not an explanation of your process.
- Preserve the facts provided by the user.
- Do not invent names, dates, promises, attachments, or completed actions.
- If an important detail is missing, use a clear placeholder such as [Name] or [Date].
- For Generate a Reply, respond to the incoming message while respecting the user's additional points.
- Do not claim that the email was sent.
- Return a useful subject line when appropriate, followed by the email body.
"""

## 4. Optional live LLM connection

If you have an API key, enter it when prompted. The key is kept in this notebook session and is not written to a file. Without a key, the notebook uses a template-based demo so you can still test the workflow.

In [ ]:
def get_api_key() -> str:
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key:
        key = getpass("Enter API key (input hidden), or leave blank for demo mode: ").strip()
    return key

def generate_with_llm(prompt: str, api_key: str, model: str = MODEL_NAME) -> str:
    """Generate a draft through the OpenAI Python SDK."""
    if not api_key:
        raise ValueError("No API key supplied.")
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    response = client.responses.create(
        model=model,
        input=prompt,
    )
    return response.output_text.strip()

## 5. Demo mode (no API key required)

In [ ]:
def generate_demo(mode: str, context: str, tone: str, length: str,
                  subject: Optional[str] = None, key_points: Optional[str] = None) -> str:
    """A predictable local example, not an actual AI-generated response."""
    points = (key_points or "").strip()
    greeting = "Hello," if tone in {"Professional", "Formal", "Concise"} else "Hi,"
    if mode == "Generate a Reply":
        body = (
            f"{greeting}\\n\\nThank you for your email. "
            "I appreciate you reaching out. "
            + (f"{points} " if points else "")
            + "Please let me know if you need any further information.\\n\\nBest regards,\\n[Your Name]"
        )
        title = subject.strip() if subject and subject.strip() else "Re: Your Email"
    else:
        body = (
            f"{greeting}\\n\\nI am writing regarding the following: {context.strip()}\\n\\n"
            + (f"{points}\\n\\n" if points else "")
            + "Please let me know if you have any questions.\\n\\nBest regards,\\n[Your Name]"
        )
        title = subject.strip() if subject and subject.strip() else "Email Subject"
    if length == "Short":
        body = body.split("\\n\\n")[0] + "\\n\\n" + (points or "Please let me know. ") + "\\n\\nBest regards,\\n[Your Name]"
    return f"Subject: {title}\\n\\n{body}"

def generate_email(mode: str, context: str, tone: str = "Professional",
                   length: str = "Standard", subject: Optional[str] = None,
                   key_points: Optional[str] = None, api_key: Optional[str] = None) -> str:
    prompt = build_email_prompt(mode, context, tone, length, subject, key_points)
    key = api_key if api_key is not None else API_KEY
    if key:
        return generate_with_llm(prompt, key)
    return generate_demo(mode, context, tone, length, subject, key_points)

## 6. Try it out

Edit the values in this cell and run it. To use a live model, configure `OPENAI_API_KEY` or pass an API key to `generate_email`.

In [ ]:
mode = "Generate a Reply"  # or "Draft an Email"
context = """Hello,
Could you please confirm whether our project review meeting is scheduled for Thursday?
Thank you,
Priya"""
tone = "Professional"
length = "Short"
subject = None
key_points = "Say that I will confirm the meeting time by tomorrow."

draft = generate_email(
    mode=mode,
    context=context,
    tone=tone,
    length=length,
    subject=subject,
    key_points=key_points,
)
print(draft)

## 7. Compare tone options

This simple table helps document the tone choices. The model may interpret tone labels somewhat differently, so review every draft.

In [ ]:
tone_guide = pd.DataFrame([
    {"Tone": "Professional", "Use case": "Workplace communication", "Style": "Clear, respectful, balanced"},
    {"Tone": "Friendly", "Use case": "Familiar colleagues", "Style": "Warm and conversational"},
    {"Tone": "Formal", "Use case": "Official requests", "Style": "Polished and structured"},
    {"Tone": "Concise", "Use case": "Quick replies", "Style": "Direct and brief"},
    {"Tone": "Appreciative", "Use case": "Thanks and acknowledgements", "Style": "Grateful and considerate"},
    {"Tone": "Persuasive", "Use case": "Proposals and requests", "Style": "Reasoned and courteous"},
])
tone_guide

## 8. Basic validation tests

In [ ]:
# Tests for prompt validation. These should pass without making an API request.
assert "Task: Draft an Email" in build_email_prompt(
    "Draft an Email", "Ask for a project update", tone="Professional", length="Short"
)

try:
    build_email_prompt("Unknown Mode", "Some context")
except ValueError:
    pass
else:
    raise AssertionError("Invalid mode should raise ValueError")

try:
    build_email_prompt("Draft an Email", "   ")
except ValueError:
    pass
else:
    raise AssertionError("Empty context should raise ValueError")

print("All validation tests passed.")

## 9. Safety and privacy checklist

Before using a generated email:
- [ ] Confirm recipient, names, dates, and facts.
- [ ] Remove any unsupported claims or commitments.
- [ ] Check the tone and whether the message says what you intend.
- [ ] Do not include confidential information unless your AI provider and environment are approved for it.
- [ ] Never commit API keys or private email data to GitHub.
- [ ] Copy the reviewed draft to your email client and send it yourself.

## 🔭 Suggested next improvements

- Add a Streamlit interface with draft/reply modes.
- Add a subject-line-only option.
- Add email-thread summarization.
- Add tests that mock the model API.
- Add optional Gmail integration only with explicit authorization and a review-before-send step.

**Note:** The notebook currently creates drafts only. It does not connect to an email account or send messages.